In [6]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import os
import seaborn as sns  

from dotenv import load_dotenv; load_dotenv()


PARALLEL_NEBULA_SCRIPT_PATH = os.getenv("PARALLEL_NEBULA_SCRIPT")

TISSUE = "Striatum"
TYPE_DEG = "baseline_nebula" #PCA_nebula, baseline_nebula. gradient_score_nebula
ADATA_PATH = f"/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/{TISSUE}/{TISSUE}_combined_QC_mmc_ct-cured.h5ad"

# name of cell tyoe vauble annotaton to use in this analsys
CT_FOR_DEG_VARIABLE = "Group_name" #"Group_name",  "spn_type"
# name of Sample varibale
SAMPLE_VARIABLE = "donor_id"
# variable to test if differtially epxresseda
CONTRAST_VARIABLE = "condition"
# Level of contrat varibale to use as baseline
CONTRAST_BASELINE = "Control"
# Level of contrat varibale to use as stimulated
CONTRAST_STIM = "XDP"

# Covariates to use for stat test
COVARIATES_FOR_DEG = ["age_of_death", "sex", "pct_counts_mt", "frac_intronic", "cohort",
                      ]
# libraru size col for  enbula
LIBRARY_SIZE_COL = "total_counts"

# # mitluple etst correction pvalue thr
# ALPHA_MULTIPLE_TEST = 0.05
# LOGFC_THR=0.1
# PVAL_THR=0.05

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
adata = sc.read_h5ad(ADATA_PATH)#, backed="r")
adata.obs.columns

###########################
#ATTENTION: use genenames for DEG
adata.var.index = adata.var["gene_symbol"]
###########################

In [11]:


controls = adata[adata.obs[CONTRAST_VARIABLE] == CONTRAST_BASELINE].copy()

In [12]:
controls

AnnData object with n_obs × n_vars = 86323 × 38601
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'barcode', 'bcl', 'rna_index', 'library', 'library__barcode', 'frac_mito', 'mol_info_nUMI', 'mol_info_nRead', 'frac_intronic', 'donor_id', 'vireo_prob_max', 'vireo_prob_doublet', 'vireo_n_vars', 'vireo_best_singlet', 'vireo_best_doublet', 'vireo_doublet_logLikRatio', 'dropsift_frac_contamination', 'dropsift_training_label_is_cell', 'dropsift_empty_gene_module_score', 'dropsift_is_cell', 'dropsift_is_cell_prob', 'cell_class', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'tissue', 'broad_original_cell_type', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'brain_bank', 'cohort', 'condition', 'sex', 'age_of_ons

In [15]:
# Assuming controls is already defined as:
# controls = adata[adata.obs[CONTRAST_VARIABLE] == CONTRAST_BASELINE].copy()

CT_COLUMN = "ct_for_deg"
CT = "STR_D1_Matrix_MSN"

LOGFC_THR = -0.5
PADJ_THR = 0.05

# Make sure X is lognorm
controls.X = controls.layers["log1p_norm"].copy()

# Rank genes: each CT vs rest
sc.tl.rank_genes_groups(
    controls,
    groupby=CT_COLUMN,
    method='wilcoxon',
    reference='rest',
)

# Extract results for CT of interest, filter for low expressed
results = sc.get.rank_genes_groups_df(controls, group=CT)

specifically_low = results[
    (results['logfoldchanges'] < LOGFC_THR) &
    (results['pvals_adj'] < PADJ_THR)
].sort_values('logfoldchanges')

print(f"Specifically low genes in {CT}: {len(specifically_low)}")
print(specifically_low.head(20))

KeyboardInterrupt: 